# TrendMart — Sales & Revenue Analytics System
**Project:** Data-Driven Strategy for a Small Retail Business  
**Scope:** ETL Pipeline · SQL/MySQL Data Warehouse · Customer Segmentation · Predictive Modeling  
**Visualization:** Power BI (connected directly to MySQL — no charts in this notebook)

---
### Notebook Structure
| # | Section |
|---|---|
| 1 | Environment Setup |
| 2 | ETL Pipeline — Extract, Transform, Load |
| 3 | Key Business KPIs |
| 4 | Sales & Revenue Aggregations |
| 5 | Discount & Promotion Analysis |
| 6 | Time-of-Day & Hourly Patterns |
| 7 | Product Performance & Pareto Analysis |
| 8 | Customer Segmentation — RFM + K-Means |
| 9 | Customer Lifetime Value (CLV) |
| 10 | Cohort Retention Analysis |
| 11 | Sales Forecasting — SARIMA |
| 12 | Data Warehouse — Star Schema Load to MySQL |
| 13 | SQL Analytical Queries |
| 14 | Executive Summary & Recommendations |

---
## 1. Environment Setup

In [25]:
import logging
import warnings
import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')
log = logging.getLogger('TrendMart')

log.info('Environment ready.')

[INFO] Environment ready.


---
## 2. ETL Pipeline — Extract, Transform, Load

In [26]:
# ── EXTRACT ───────────────────────────────────────────────────────────────────
def extract():
    sales     = pd.read_csv('sales_updated.csv')
    customers = pd.read_csv('customers_updated.csv')
    log.info(f'Extracted  ->  Sales: {sales.shape}  |  Customers: {customers.shape}')
    return sales, customers

sales_raw, customers_raw = extract()

[INFO] Extracted  ->  Sales: (2705, 9)  |  Customers: (1000, 6)


In [27]:
# ── TRANSFORM ─────────────────────────────────────────────────────────────────
def transform_sales(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.copy()

    # --- Data quality ---
    df['Date'] = pd.to_datetime(df['Date'])
    df.drop_duplicates(subset='Transaction_ID', inplace=True)
    df.dropna(subset=['Customer_ID', 'Total_Amount'], inplace=True)
    df = df[df['Total_Amount'] > 0]

    # --- Temporal features ---
    df['Year']       = df['Date'].dt.year
    df['Month']      = df['Date'].dt.month
    df['Month_Name'] = df['Date'].dt.strftime('%b')
    df['Quarter']    = df['Date'].dt.quarter
    df['Weekday']    = df['Date'].dt.day_name()
    df['Hour']       = df['Date'].dt.hour
    df['YearMonth']  = df['Date'].dt.to_period('M')

    # --- Time-of-day segment ---
    bins   = [-1, 5, 11, 16, 20, 23]
    labels = ['Night (0-5)', 'Morning (6-11)', 'Afternoon (12-16)',
            'Evening (17-20)', 'Night (21-23)']
    df['Time_Of_Day'] = pd.cut(df['Hour'], bins=bins, labels=labels)

    # --- Discount feature engineering ---
    # NOTE: No discount fields in raw data.
    # Proxy: Qty >= 4 → 5-20% bulk discount | Qty == 3 → 1-5% | Qty <= 2 → 0
    # FIX: pre-generate both random arrays immediately after seed so both tiers
    # draw from the same reproducible sequence.
    np.random.seed(42)
    rand_bulk  = np.random.uniform(0.05, 0.20, len(df))
    rand_small = np.random.uniform(0.01, 0.05, len(df))

    df['Gross_Amount']   = df['Unit_Price'] * df['Quantity']
    df['Discount_Rate']  = np.where(
        df['Quantity'] >= 4, rand_bulk,
        np.where(df['Quantity'] == 3, rand_small, 0)
    ).round(4)
    df['Discount_Amount'] = (df['Gross_Amount'] * df['Discount_Rate']).round(2)
    df['Net_Amount']      = (df['Gross_Amount'] - df['Discount_Amount']).round(2)
    df['Promotion_Flag']  = (df['Discount_Rate'] > 0).astype(int)

    # --- Margin features (60% COGS assumed — no cost data in dataset) ---
    COST_RATE           = 0.60
    df['Gross_Profit']  = (df['Gross_Amount'] * (1 - COST_RATE)).round(2)
    df['Net_Profit']    = (df['Net_Amount'] - df['Gross_Amount'] * COST_RATE).round(2)

    log.info(f'Sales transform complete  ->  {df.shape[0]:,} rows, {df.shape[1]} cols')
    return df


def transform_customers(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.copy()
    df.drop_duplicates(subset='Customer_ID', inplace=True)
    df['Age_Group'] = pd.cut(
        df['Age'], bins=[0, 25, 35, 50, 100],
        labels=['Gen-Z (<=25)', 'Millennial (26-35)', 'Gen-X (36-50)', 'Boomer (50+)']
    ).astype(str)
    log.info(f'Customer transform complete  ->  {df.shape[0]:,} rows')
    return df


sales     = transform_sales(sales_raw)
customers = transform_customers(customers_raw)

# Master merged analytical frame
df = sales.merge(customers, on='Customer_ID', how='left')
log.info(f'Merged dataset  ->  {df.shape[0]:,} rows x {df.shape[1]} cols')
df.head(3)


[INFO] Sales transform complete  ->  2,705 rows, 24 cols
[INFO] Customer transform complete  ->  1,000 rows
[INFO] Merged dataset  ->  2,705 rows x 30 cols


,Transaction_ID,Date,Customer_ID,Product_Category,Product_Name,Quantity,Unit_Price,Total_Amount,Payment_Method,Year,...,Net_Amount,Promotion_Flag,Gross_Profit,Net_Profit,Name,Age,Gender,Location,Total_Spent,Age_Group
0,f3daa723,2024-01-01 00:00:00,0583d6,Clothing,Sofa,4,412,1648,Credit Card,2024,...,1472.98,1,659.2,484.18,Kevin Hanna,22,Female,Los Angeles,4179,Gen-Z (<=25)
1,9e5061b5,2024-01-01 01:00:00,4c0f1b,Home Decor,Novel,3,201,603,Cash,2024,...,594.08,1,241.2,232.28,Marisa Sullivan,60,Other,Los Angeles,1683,Boomer (50+)
2,a5da9276,2024-01-01 02:00:00,00b49a,Home Decor,T-Shirt,1,480,480,Credit Card,2024,...,480.00,0,192.0,192.00,Shawn Gonzales,45,Female,Houston,358,Gen-X (36-50)


In [28]:
# ── DATA QUALITY REPORT ───────────────────────────────────────────────────────
print('Null counts after cleaning:')
nulls = df.isnull().sum()
print(nulls[nulls > 0].to_string() if nulls.any() else '  None — dataset is clean')
print(f'\nDate range  : {df["Date"].min().date()}  to  {df["Date"].max().date()}')
print(f'Transactions: {df["Transaction_ID"].nunique():,}')
print(f'Customers   : {df["Customer_ID"].nunique():,}')
print(f'Products    : {df["Product_Name"].nunique()}')
print(f'Categories  : {df["Product_Category"].nunique()}')

Null counts after cleaning:
  None — dataset is clean

Date range  : 2024-01-01  to  2024-04-14
Transactions: 2,705
Customers   : 1,000
Products    : 5
Categories  : 5


---
## 3. Key Business KPIs

In [29]:
total_revenue      = df['Net_Amount'].sum()
total_transactions = df['Transaction_ID'].nunique()
total_customers    = df['Customer_ID'].nunique()
aov                = total_revenue / total_transactions
rev_per_customer   = total_revenue / total_customers
total_discount     = df['Discount_Amount'].sum()
discount_pct       = total_discount / df['Gross_Amount'].sum() * 100
repeat_rate        = (df.groupby('Customer_ID')['Transaction_ID'].count() > 1).mean() * 100
total_units        = df['Quantity'].sum()
avg_discount_rate  = df[df['Promotion_Flag'] == 1]['Discount_Rate'].mean() * 100

kpis = {
    'Total Revenue (Net)'       : f'${total_revenue:>13,.2f}',
    'Total Gross Revenue'       : f'${df["Gross_Amount"].sum():>13,.2f}',
    'Total Transactions'        : f'{total_transactions:>14,}',
    'Total Units Sold'          : f'{total_units:>14,}',
    'Unique Customers'          : f'{total_customers:>14,}',
    'Average Order Value (AOV)' : f'${aov:>13,.2f}',
    'Revenue per Customer'      : f'${rev_per_customer:>13,.2f}',
    'Total Discounts Given'     : f'${total_discount:>13,.2f}',
    'Avg Discount Rate (promo)' : f'{avg_discount_rate:>13.2f}%',
    'Overall Discount Rate (%)'  : f'{discount_pct:>13.2f}%',
    'Repeat Purchase Rate (%)'  : f'{repeat_rate:>13.2f}%',
}

print('=' * 52)
print('          TRENDMART — BUSINESS KPIs')
print('=' * 52)
for k, v in kpis.items():
    print(f'  {k:<31} {v}')
print('=' * 52)

          TRENDMART — BUSINESS KPIs
  Total Revenue (Net)             $ 1,670,762.22
  Total Gross Revenue             $ 1,782,148.00
  Total Transactions                       2,705
  Total Units Sold                         6,927
  Unique Customers                         1,000
  Average Order Value (AOV)       $       617.66
  Revenue per Customer            $     1,670.76
  Total Discounts Given           $   111,385.78
  Avg Discount Rate (promo)                8.01%
  Overall Discount Rate (%)                6.25%
  Repeat Purchase Rate (%)                78.30%


---
## 4. Sales & Revenue Aggregations

In [30]:
# ── Monthly revenue summary ───────────────────────────────────────────────────
monthly = (df.groupby(['Year', 'Month', 'Month_Name'])
             .agg(Revenue=('Net_Amount', 'sum'),
                  Transactions=('Transaction_ID', 'count'),
                  Avg_Order=('Net_Amount', 'mean'),
                  Units_Sold=('Quantity', 'sum'))
             .reset_index().sort_values(['Year', 'Month']))
monthly['Period']     = monthly['Month_Name'] + ' ' + monthly['Year'].astype(str)
monthly['MoM_Change'] = monthly['Revenue'].diff().round(2)
monthly['MoM_Pct']    = monthly['Revenue'].pct_change().mul(100).round(2)

print('Monthly Revenue Summary:')
print(monthly[['Period', 'Revenue', 'Transactions', 'Avg_Order',
               'Units_Sold', 'MoM_Change', 'MoM_Pct']].to_string(index=False))

Monthly Revenue Summary:
  Period   Revenue  Transactions  Avg_Order  Units_Sold  MoM_Change  MoM_Pct
Jan 2024 496221.99           810 612.619741        2054         NaN      NaN
Feb 2024 448407.12           741 605.137814        1885   -47814.87    -9.64
Mar 2024 501444.78           810 619.067630        2062    53037.66    11.83
Apr 2024 224688.33           344 653.163750         926  -276756.45   -55.19


In [31]:
# ── Revenue by category ───────────────────────────────────────────────────────
cat = (df.groupby('Product_Category')
         .agg(Revenue=('Net_Amount', 'sum'),
              Transactions=('Transaction_ID', 'count'),
              Avg_Order=('Net_Amount', 'mean'),
              Units_Sold=('Quantity', 'sum'))
         .sort_values('Revenue', ascending=False))
cat['Revenue_Share_%'] = (cat['Revenue'] / cat['Revenue'].sum() * 100).round(2)

print('Revenue by Category:')
print(cat.round(2).to_string())

Revenue by Category:
                    Revenue  Transactions  Avg_Order  Units_Sold  Revenue_Share_%
Product_Category                                                                 
Clothing          347248.73           539     644.25        1420            20.78
Electronics       343520.33           561     612.34        1437            20.56
Home Decor        336694.97           550     612.17        1388            20.15
Books             321717.23           532     604.73        1339            19.26
Beauty            321580.96           523     614.88        1343            19.25


In [32]:
# ── Revenue by location, gender, age group, payment ──────────────────────────
loc_rev = (df.groupby('Location')
             .agg(Revenue=('Net_Amount', 'sum'),
                  Customers=('Customer_ID', 'nunique'),
                  Transactions=('Transaction_ID', 'count'))
             .sort_values('Revenue', ascending=False))

gender_rev = (df.groupby('Gender')
                .agg(Revenue=('Net_Amount', 'sum'),
                     Customers=('Customer_ID', 'nunique'))
                .sort_values('Revenue', ascending=False))

age_rev = (df.groupby('Age_Group')
             .agg(Revenue=('Net_Amount', 'sum'),
                  Customers=('Customer_ID', 'nunique'))
             .sort_values('Revenue', ascending=False))

payment_rev = (df.groupby('Payment_Method')
                 .agg(Revenue=('Net_Amount', 'sum'),
                      Transactions=('Transaction_ID', 'count'),
                      Avg_Order=('Net_Amount', 'mean'))
                 .sort_values('Revenue', ascending=False))
payment_rev['Revenue_Share_%'] = (payment_rev['Revenue'] / payment_rev['Revenue'].sum() * 100).round(2)

print('Revenue by Location:'); print(loc_rev.round(2).to_string())
print('\nRevenue by Gender:'); print(gender_rev.round(2).to_string())
print('\nRevenue by Age Group:'); print(age_rev.round(2).to_string())
print('\nRevenue by Payment Method:'); print(payment_rev.round(2).to_string())

Revenue by Location:
               Revenue  Customers  Transactions
Location                                       
New York     368599.99        207           584
Los Angeles  348187.25        197           559
Chicago      329910.82        209           550
Houston      316089.22        199           507
Miami        307974.94        188           505

Revenue by Gender:
          Revenue  Customers
Gender                      
Male    559980.66        330
Female  556164.33        335
Other   554617.23        335

Revenue by Age Group:
                      Revenue  Customers
Age_Group                               
Boomer (50+)        624783.22        356
Gen-X (36-50)       463844.67        301
Millennial (26-35)  328804.68        199
Gen-Z (<=25)        253329.65        144

Revenue by Payment Method:
                  Revenue  Transactions  Avg_Order  Revenue_Share_%
Payment_Method                                                     
Cash            425464.63           684     6

---
## 5. Discount & Promotion Analysis
> **Note:** No discount fields exist in the raw dataset. Discount variables (`Discount_Rate`, `Discount_Amount`, `Promotion_Flag`) are derived from purchase quantity as a realistic business proxy. COGS is assumed at 60%.

In [33]:
# ── Promo vs no-promo comparison ──────────────────────────────────────────────
disc_summary = (df.groupby('Promotion_Flag')
                  .agg(Transactions   =('Transaction_ID', 'count'),
                       Avg_Gross      =('Gross_Amount', 'mean'),
                       Avg_Net        =('Net_Amount', 'mean'),
                       Avg_Discount   =('Discount_Amount', 'mean'),
                       Total_Revenue  =('Net_Amount', 'sum'),
                       Total_Discount =('Discount_Amount', 'sum'))
                  .rename(index={0: 'No Promotion', 1: 'Promotion Applied'}))
disc_summary['Revenue_Share_%'] = (
    disc_summary['Total_Revenue'] / disc_summary['Total_Revenue'].sum() * 100).round(2)

print('Promotion Impact Summary:')
print(disc_summary.round(2).to_string())

Promotion Impact Summary:
                   Transactions  Avg_Gross  Avg_Net  Avg_Discount  Total_Revenue  Total_Discount  Revenue_Share_%
Promotion_Flag                                                                                                   
No Promotion               1316     384.56   384.56          0.00      506087.00            0.00            30.29
Promotion Applied          1389     918.69   838.50         80.19     1164675.22       111385.78            69.71


In [34]:
# ── Discount by category & margin erosion ────────────────────────────────────
cat_disc = (df[df['Promotion_Flag'] == 1]
              .groupby('Product_Category')
              .agg(Avg_Discount_Rate  =('Discount_Rate', 'mean'),
                   Total_Discount     =('Discount_Amount', 'sum'),
                   Promo_Transactions =('Transaction_ID', 'count'),
                   Promo_Revenue      =('Net_Amount', 'sum'))
              .sort_values('Total_Discount', ascending=False))
cat_disc['Avg_Discount_Rate_%'] = (cat_disc['Avg_Discount_Rate'] * 100).round(2)

print('Discount by Category (promoted transactions only):')
print(cat_disc[['Avg_Discount_Rate_%', 'Total_Discount',
                'Promo_Transactions', 'Promo_Revenue']].round(2).to_string())

margin = (df.groupby('Promotion_Flag')
            .agg(Avg_Gross_Profit=('Gross_Profit', 'mean'),
                 Avg_Net_Profit  =('Net_Profit', 'mean'),
                 Total_Net_Profit=('Net_Profit', 'sum'))
            .rename(index={0: 'No Promotion', 1: 'Promotion Applied'}))

print('\nMargin Erosion from Promotions (60% COGS assumed):')
print(margin.round(2).to_string())

erosion = margin.loc['No Promotion', 'Avg_Net_Profit'] - margin.loc['Promotion Applied', 'Avg_Net_Profit']
print(f'\n  Avg margin lost per promoted transaction: ${erosion:.2f}')

Discount by Category (promoted transactions only):
                  Avg_Discount_Rate_%  Total_Discount  Promo_Transactions  Promo_Revenue
Product_Category                                                                        
Clothing                         8.21        24534.27                 289      254903.73
Electronics                      8.26        23284.67                 290      237566.33
Home Decor                       8.09        22209.03                 271      233707.97
Beauty                           7.83        20750.04                 274      219572.96
Books                            7.64        20607.77                 265      218924.23

Margin Erosion from Promotions (60% COGS assumed):
                   Avg_Gross_Profit  Avg_Net_Profit  Total_Net_Profit
Promotion_Flag                                                       
No Promotion                 153.83          153.83         202434.80
Promotion Applied            367.48          287.28         3990

---
## 6. Time-of-Day & Hourly Patterns

In [35]:
# ── Hourly aggregation ────────────────────────────────────────────────────────
hourly = (df.groupby('Hour')
            .agg(Revenue=('Net_Amount', 'sum'),
                 Transactions=('Transaction_ID', 'count'),
                 Avg_Order=('Net_Amount', 'mean'))
            .reset_index())

peak_hour         = hourly.loc[hourly['Transactions'].idxmax(), 'Hour']
peak_revenue_hour = hourly.loc[hourly['Revenue'].idxmax(), 'Hour']

print('Hourly Sales Summary:')
print(hourly.round(2).to_string(index=False))
print(f'\n  Peak transaction hour : {peak_hour}:00')
print(f'  Peak revenue hour     : {peak_revenue_hour}:00')

Hourly Sales Summary:
 Hour  Revenue  Transactions  Avg_Order
    0 67490.58           110     613.55
    1 86523.53           118     733.25
    2 70372.51           114     617.30
    3 64844.15           114     568.81
    4 67690.28           111     609.82
    5 70024.53           115     608.91
    6 54916.46           105     523.01
    7 72584.68           110     659.86
    8 70222.22           109     644.24
    9 67990.50           110     618.10
   10 67117.28           112     599.26
   11 70081.61           112     625.73
   12 75672.76           119     635.91
   13 78807.23           117     673.57
   14 61839.49           108     572.59
   15 67897.61           111     611.69
   16 71415.43           115     621.00
   17 63803.52           111     574.81
   18 67380.65           116     580.87
   19 79898.44           114     700.86
   20 67594.07           112     603.52
   21 64234.96           112     573.53
   22 68864.31           111     620.40
   23 73495.42    

In [36]:
# ── Time-of-day & category breakdown ─────────────────────────────────────────
tod = (df.groupby('Time_Of_Day', observed=True)
         .agg(Revenue=('Net_Amount', 'sum'),
              Transactions=('Transaction_ID', 'count'),
              Avg_Order=('Net_Amount', 'mean'))
         .reset_index().sort_values('Revenue', ascending=False))
tod['Revenue_Share_%'] = (tod['Revenue'] / tod['Revenue'].sum() * 100).round(2)

cat_tod = (df.groupby(['Time_Of_Day', 'Product_Category'], observed=True)
             .agg(Revenue=('Net_Amount', 'sum'),
                  Transactions=('Transaction_ID', 'count'))
             .reset_index().sort_values('Revenue', ascending=False))

print('Revenue by Time-of-Day Segment:')
print(tod.round(2).to_string(index=False))
print('\nCategory Revenue by Time of Day:')
print(cat_tod.round(2).to_string(index=False))

Revenue by Time-of-Day Segment:
      Time_Of_Day   Revenue  Transactions  Avg_Order  Revenue_Share_%
      Night (0-5) 426945.58           682     626.02            25.55
   Morning (6-11) 402912.75           658     612.33            24.12
Afternoon (12-16) 355632.52           570     623.92            21.29
  Evening (17-20) 278676.68           453     615.18            16.68
    Night (21-23) 206594.69           342     604.08            12.37

Category Revenue by Time of Day:
      Time_Of_Day Product_Category  Revenue  Transactions
      Night (0-5)         Clothing 92202.54           153
   Morning (6-11)         Clothing 88697.06           133
      Night (0-5)      Electronics 85900.89           138
   Morning (6-11)           Beauty 85106.26           141
      Night (0-5)           Beauty 84793.90           129
      Night (0-5)       Home Decor 84628.71           140
   Morning (6-11)       Home Decor 81249.39           130
Afternoon (12-16)       Home Decor 80515.49       

---
## 7. Product Performance & Pareto Analysis

In [37]:
def pareto_analysis(df: pd.DataFrame) -> pd.DataFrame:
    prod = (df.groupby('Product_Name')
              .agg(Revenue     =('Net_Amount', 'sum'),
                   Transactions=('Transaction_ID', 'count'),
                   Units_Sold  =('Quantity', 'sum'),
                   Avg_Price   =('Unit_Price', 'mean'))
              .sort_values('Revenue', ascending=False)
              .reset_index())
    prod['Cumulative_Revenue'] = prod['Revenue'].cumsum()
    prod['Cumulative_%']       = (prod['Cumulative_Revenue'] / prod['Revenue'].sum() * 100).round(2)
    prod['Revenue_Share_%']    = (prod['Revenue'] / prod['Revenue'].sum() * 100).round(2)
    return prod

prod_perf = pareto_analysis(df)

print('Product Performance & Pareto Analysis:')
print(prod_perf[['Product_Name', 'Revenue', 'Revenue_Share_%',
                 'Cumulative_%', 'Transactions', 'Units_Sold']].to_string(index=False))

top_products = prod_perf[prod_perf['Cumulative_%'] <= 80]['Product_Name'].tolist()
print(f'\n  Products driving 80% of revenue : {top_products}')
print(f'  Lowest-revenue product          : {prod_perf.iloc[-1]["Product_Name"]}')

Product Performance & Pareto Analysis:
Product_Name   Revenue  Revenue_Share_%  Cumulative_%  Transactions  Units_Sold
        Sofa 363388.98            21.75         21.75           570        1485
       Novel 336529.01            20.14         41.89           562        1405
    Lipstick 330605.83            19.79         61.68           528        1383
      Laptop 327726.10            19.62         81.30           524        1333
     T-Shirt 312512.30            18.70        100.00           521        1321

  Products driving 80% of revenue : ['Sofa', 'Novel', 'Lipstick']
  Lowest-revenue product          : T-Shirt


---
## 8. Customer Segmentation — RFM + K-Means

In [ ]:
def build_rfm(df: pd.DataFrame) -> pd.DataFrame:
    snapshot = df['Date'].max() + pd.Timedelta(days=1)
    rfm = (df.groupby('Customer_ID').agg(
        Recency   =('Date', lambda x: (snapshot - x.max()).days),
        Frequency =('Transaction_ID', 'count'),
        Monetary  =('Net_Amount', 'sum')
    ).reset_index())
    log.info(f'RFM built  ->  {len(rfm):,} customers')
    return rfm

rfm = build_rfm(df)

# Standardize for clustering
scaler     = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

# Elbow method — find optimal k
inertia = {k: KMeans(n_clusters=k, random_state=42, n_init=10).fit(rfm_scaled).inertia_
        for k in range(2, 8)}
print('Elbow Method — Inertia by k:')
for k, v in inertia.items():
    print(f'  k={k}  inertia={v:,.1f}')
print('  -> k=3 selected')

[INFO] RFM built  ->  1,000 customers


Elbow Method — Inertia by k:
  k=2  inertia=1,701.9
  k=3  inertia=1,172.2
  k=4  inertia=913.7
  k=5  inertia=781.8
  k=6  inertia=707.7
  k=7  inertia=642.7
  -> k=3 selected


In [ ]:
# K-Means clustering (k=3) 
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(rfm_scaled)

# Label by descending monetary value
mon_rank  = rfm.groupby('Cluster')['Monetary'].mean().rank(ascending=False).astype(int)
label_map = {c: ['High-Value', 'Frequent Buyer', 'Occasional Buyer'][mon_rank[c]-1]
            for c in mon_rank.index}
rfm['Segment'] = rfm['Cluster'].map(label_map)

# Deep cluster profile
cluster_profile = (rfm.groupby('Segment')
                      .agg(Count        =('Customer_ID', 'count'),
                           Avg_Recency  =('Recency', 'mean'),
                           Avg_Frequency=('Frequency', 'mean'),
                           Avg_Monetary =('Monetary', 'mean'),
                           Total_Revenue=('Monetary', 'sum'))
                      .round(2)
                      .sort_values('Avg_Monetary', ascending=False))
cluster_profile['Revenue_Share_%'] = (
    cluster_profile['Total_Revenue'] / cluster_profile['Total_Revenue'].sum() * 100).round(2)

print('Cluster Deep Profile:')
print(cluster_profile.to_string())

strategies = {
    'High-Value'      : 'VIP loyalty programme | Exclusive early access | Premium bundles',
    'Frequent Buyer'  : 'Volume discounts | Subscription model | Cross-sell promotions',
    'Occasional Buyer': 'Re-engagement campaigns | Flash sales | Personalised offers'
}
print('\nSegment Strategy:')
for seg, strat in strategies.items():
    print(f'  [{seg:<18}] {strat}')

Cluster Deep Profile:
                  Count  Avg_Recency  Avg_Frequency  Avg_Monetary  Total_Revenue  Revenue_Share_%
Segment                                                                                          
High-Value          277        19.68           4.38       3053.04      845692.01            50.62
Frequent Buyer      449        19.69           2.32       1273.05      571598.74            34.21
Occasional Buyer    274        67.87           1.65        925.08      253471.47            15.17

Segment Strategy:
  [High-Value        ] VIP loyalty programme | Exclusive early access | Premium bundles
  [Frequent Buyer    ] Volume discounts | Subscription model | Cross-sell promotions
  [Occasional Buyer  ] Re-engagement campaigns | Flash sales | Personalised offers


---
## 9. Customer Lifetime Value (CLV)
> **Assumptions applied (no actuals available in dataset):**  
> Margin Rate = 30% | Churn Rate = 20% | Discount Rate = 10%

In [40]:
MARGIN_RATE   = 0.30
CHURN_RATE    = 0.20
DISCOUNT_RATE = 0.10

rfm['Avg_Order_Value'] = (rfm['Monetary'] / rfm['Frequency']).round(2)
rfm['CLV'] = (
    (rfm['Avg_Order_Value'] * rfm['Frequency'] * MARGIN_RATE)
    / (DISCOUNT_RATE + CHURN_RATE)
).round(2)

clv_summary = rfm.groupby('Segment')['CLV'].agg(
    Min='min', Mean='mean', Max='max', Total='sum'
).round(2)

print('CLV Summary by Segment:')
print(clv_summary.to_string())
print(f'\n  Overall Avg CLV : ${rfm["CLV"].mean():,.2f}')
print(f'  Total CLV Pool  : ${rfm["CLV"].sum():,.2f}')

CLV Summary by Segment:
                     Min     Mean      Max      Total
Segment                                              
Frequent Buyer     34.00  1273.05  3083.30  571598.78
High-Value        970.40  3053.04  6881.40  845692.14
Occasional Buyer   31.81   925.08  3273.33  253471.41

  Overall Avg CLV : $1,670.76
  Total CLV Pool  : $1,670,762.33


---
## 10. Cohort Retention Analysis

In [41]:
def cohort_analysis(df: pd.DataFrame):
    cohort_df = df[['Customer_ID', 'YearMonth']].copy()
    cohort_df['Cohort']        = cohort_df.groupby('Customer_ID')['YearMonth'].transform('min')
    cohort_df['Period_Number'] = (
        cohort_df['YearMonth'] - cohort_df['Cohort']
    ).apply(lambda x: x.n)
    counts    = (cohort_df.groupby(['Cohort', 'Period_Number'])['Customer_ID']
                          .nunique().reset_index())
    pivot     = counts.pivot(index='Cohort', columns='Period_Number', values='Customer_ID')
    retention = pivot.divide(pivot.iloc[:, 0], axis=0).round(4) * 100
    return retention, pivot

retention, cohort_counts = cohort_analysis(df)

print('Cohort Retention Rates (%) — rows: first-purchase month, cols: months since first purchase:')
print(retention.round(1).to_string())
print('\nCohort Customer Counts (absolute):')
print(cohort_counts.to_string())

# Flatten for MySQL
retention_flat = retention.reset_index().melt(
    id_vars='Cohort', var_name='Period_Number', value_name='Retention_Pct'
)
retention_flat['Cohort']        = retention_flat['Cohort'].astype(str)
retention_flat['Retention_Pct'] = retention_flat['Retention_Pct'].round(2)

log.info('Cohort analysis complete.')

[INFO] Cohort analysis complete.


Cohort Retention Rates (%) — rows: first-purchase month, cols: months since first purchase:
Period_Number      0     1     2     3
Cohort                                
2024-01        100.0  48.9  52.5  25.4
2024-02        100.0  54.2  27.6   NaN
2024-03        100.0  28.9   NaN   NaN
2024-04        100.0   NaN   NaN   NaN

Cohort Customer Counts (absolute):
Period_Number      0      1      2      3
Cohort                                   
2024-01        564.0  276.0  296.0  143.0
2024-02        264.0  143.0   73.0    NaN
2024-03        135.0   39.0    NaN    NaN
2024-04         37.0    NaN    NaN    NaN


---
## 11. Sales Forecasting — SARIMA

In [42]:
# ── Monthly time series & seasonal decomposition ──────────────────────────────
ts = (df.groupby('YearMonth')['Net_Amount'].sum().sort_index())
ts.index = ts.index.to_timestamp()
ts.index.freq = 'MS'

print('Monthly Revenue Time Series:')
print(ts.round(2).to_string())

if len(ts) >= 4:
    decomp = seasonal_decompose(ts, model='additive', period=2)
    decomp_df = pd.DataFrame({
        'Observed' : decomp.observed,
        'Trend'    : decomp.trend,
        'Seasonal' : decomp.seasonal,
        'Residual' : decomp.resid
    })
    print('\nSeasonal Decomposition Components:')
    print(decomp_df.round(2).to_string())

Monthly Revenue Time Series:
YearMonth
2024-01-01    496221.99
2024-02-01    448407.12
2024-03-01    501444.78
2024-04-01    224688.33
Freq: MS

Seasonal Decomposition Components:
             Observed      Trend  Seasonal  Residual
YearMonth                                           
2024-01-01  496221.99        NaN  53830.83       NaN
2024-02-01  448407.12  473620.25 -53830.83   28617.7
2024-03-01  501444.78  418996.25  53830.83   28617.7
2024-04-01  224688.33        NaN -53830.83       NaN


In [ ]:
#  SARIMA fit & 3-month forecast 
FORECAST_MONTHS = 3
train = ts[:-1] if len(ts) > 2 else ts

sarima = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(0, 0, 0, 0),
    enforce_stationarity=False,
    enforce_invertibility=False
).fit(disp=False)

# Model evaluation metrics 
in_sample = sarima.fittedvalues
mae  = mean_absolute_error(train, in_sample)
rmse = np.sqrt(mean_squared_error(train, in_sample))
mape = np.mean(np.abs((train - in_sample) / train)) * 100

print('SARIMA Model Evaluation Metrics:')
print(f'  MAE  : {mae:>10,.2f} USD')
print(f'  RMSE : {rmse:>10,.2f} USD')
print(f'  MAPE : {mape:>10.2f} %')

#  Forecast output 
forecast   = sarima.get_forecast(steps=FORECAST_MONTHS)
fcast_mean = forecast.predicted_mean
fcast_ci   = forecast.conf_int()

print(f'\n{FORECAST_MONTHS}-Month Revenue Forecast:')
for dt, val in fcast_mean.items():
    lo = fcast_ci.loc[dt].iloc[0]
    hi = fcast_ci.loc[dt].iloc[1]
    print(f'  {dt.strftime("%b %Y")} : ${val:>10,.2f}  (95% CI: ${lo:,.2f} - ${hi:,.2f})')

# ── FIX: store Month as Python date object so sqlalchemy writes MySQL DATE ─────
# Power BI will then import this column as Date type automatically —
# no manual type conversion in Power Query needed.
forecast_df = pd.DataFrame({
    'Month'           : [dt.date() for dt in fcast_mean.index],
    'Forecast_Revenue': fcast_mean.values.round(2),
    'CI_Lower'        : fcast_ci.iloc[:, 0].values.round(2),
    'CI_Upper'        : fcast_ci.iloc[:, 1].values.round(2)
})
log.info(f'Forecast table built  ->  {len(forecast_df)} rows  (Month stored as Python date)')


[INFO] Forecast table built  ->  3 rows  (Month stored as Python date)


SARIMA Model Evaluation Metrics:
  MAE  : 199,024.84 USD
  RMSE : 289,445.17 USD
  MAPE :      40.41 %

3-Month Revenue Forecast:
  Apr 2024 : $501,444.78  (95% CI: $420,747.43 - $582,142.13)
  May 2024 : $501,444.78  (95% CI: $387,321.50 - $615,568.06)
  Jun 2024 : $501,444.78  (95% CI: $361,672.87 - $641,216.69)


---
## 12. Data Warehouse — Star Schema Load to MySQL

> **Configure MySQL credentials below before running.**

```
Star Schema:
  fact_sales ──── dim_customer
             ──── dim_product
             ──── dim_date
             ──── dim_payment

Analytical Tables (loaded separately for Power BI):
  customer_segments        (RFM scores + K-Means label + CLV)
  monthly_revenue_summary  (aggregated monthly KPIs + MoM)
  cohort_retention         (retention % matrix for Power BI heatmap)
  sales_forecast           (SARIMA 3-month forecast + 95% CI)
```

In [44]:
DB_USER = 'root'
DB_PASS = 'Monika#80'
DB_HOST = 'localhost'
DB_PORT = '3306'
DB_NAME = 'trendmart_dw'

engine = create_engine(
    f'mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/', echo=False)
with engine.connect() as conn:
    conn.execute(text(f'CREATE DATABASE IF NOT EXISTS {DB_NAME}'))
engine = create_engine(
    f'mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}', echo=False)

log.info(f'Connected to MySQL  ->  {DB_NAME}')

[INFO] Connected to MySQL  ->  trendmart_dw


In [45]:
# ── Build all tables ──────────────────────────────────────────────────────────
#
# FIXES APPLIED IN THIS CELL:
#   1. dim_date   — 1 row per calendar date, Date as Python date → MySQL DATE
#   2. dim_product — 25 rows via groupby+median, + Product_Key surrogate
#   3. fact_sales  — Date as Python date → MySQL DATE, + Product_Key surrogate
#   4. customer_segments — Cluster column excluded
#   5. monthly_summary   — Partial_Month flag added
#
# Using Python datetime.date objects for all date columns means sqlalchemy
# writes MySQL DATE (not DATETIME/VARCHAR). Power BI reads MySQL DATE as
# Date type automatically — no Power Query type changes required at all.
# ──────────────────────────────────────────────────────────────────────────────

# ── FIX 1: dim_date — one row per calendar date, MySQL DATE type ───────────────
# Normalize strips the time component (00:00:00) before dedup.
# .dt.date converts to Python datetime.date → sqlalchemy → MySQL DATE.
# Hour and Time_Of_Day are excluded: they are transaction-level, not date-level.
dim_date = (df.assign(Date=df['Date'].dt.normalize())
              [['Date', 'Year', 'Month', 'Month_Name', 'Quarter', 'Weekday']]
              .drop_duplicates(subset='Date')
              .sort_values('Date')
              .reset_index(drop=True))
dim_date['Date_ID'] = dim_date.index + 1
dim_date['Date']    = dim_date['Date'].dt.date          # Python date → MySQL DATE
log.info(f'dim_date built  ->  {len(dim_date):,} rows  (expect 105, date range 2024-01-01 to 2024-04-14)')

# ── FIX 2: dim_product — 25 rows, one per Product+Category ────────────────────
# Original drop_duplicates on (Name, Category, Price) gave 2412 rows because
# Unit_Price varies per transaction. Groupby + median gives exactly 25 rows.
# Product_Key is a surrogate joining column — unique across all 25 rows.
# Use Product_Key (not Product_Name) to create the Power BI relationship.
dim_product = (df.groupby(['Product_Name', 'Product_Category'])
                 .agg(Unit_Price=('Unit_Price', 'median'))
                 .reset_index()
                 .reset_index()
                 .rename(columns={'index': 'Product_ID'}))
dim_product['Product_Key'] = (dim_product['Product_Name'].str.replace(' ', '_', regex=False)
                               + '__'
                               + dim_product['Product_Category'].str.replace(' ', '_', regex=False))
log.info(f'dim_product built  ->  {len(dim_product):,} rows  (expect 25, Product_Key unique: {dim_product["Product_Key"].nunique()})')

# ── dim_customer ───────────────────────────────────────────────────────────────
dim_customer = customers[['Customer_ID', 'Name', 'Age', 'Age_Group',
                           'Gender', 'Location', 'Total_Spent']].copy()

# ── dim_payment ────────────────────────────────────────────────────────────────
dim_payment = (df[['Payment_Method']].drop_duplicates()
                 .reset_index(drop=True).reset_index()
                 .rename(columns={'index': 'Payment_ID'}))

# ── FIX 3: fact_sales — Date as MySQL DATE, + Product_Key surrogate ────────────
# .dt.date converts datetime64 to Python datetime.date.
# sqlalchemy maps datetime.date → MySQL DATE (not DATETIME).
# Power BI auto-imports MySQL DATE as Date type — no manual step needed.
# Product_Key added to enable a clean unique relationship with dim_product.
fact_sales = df[['Transaction_ID', 'Date', 'Customer_ID', 'Product_Name', 'Product_Category',
                 'Payment_Method', 'Quantity', 'Unit_Price', 'Gross_Amount',
                 'Discount_Rate', 'Discount_Amount', 'Net_Amount',
                 'Gross_Profit', 'Net_Profit', 'Promotion_Flag',
                 'Year', 'Month', 'Quarter', 'Hour', 'Time_Of_Day']].copy()
fact_sales['Date']        = fact_sales['Date'].dt.date          # Python date → MySQL DATE
fact_sales['Time_Of_Day'] = fact_sales['Time_Of_Day'].astype(str)
fact_sales['Product_Key'] = (fact_sales['Product_Name'].str.replace(' ', '_', regex=False)
                              + '__'
                              + fact_sales['Product_Category'].str.replace(' ', '_', regex=False))
log.info(f'fact_sales built  ->  {len(fact_sales):,} rows')

# ── FIX 4: customer_segments — exclude internal Cluster column ─────────────────
customer_segments = rfm[['Customer_ID', 'Recency', 'Frequency',
                          'Monetary', 'Avg_Order_Value', 'Segment', 'CLV']].copy()

# ── FIX 5: monthly_summary — add Partial_Month flag ───────────────────────────
# April 2024 = only 14 days. Flag lets Power BI slicers exclude it from trends.
monthly_summary   = monthly[['Year', 'Month', 'Period', 'Revenue', 'Transactions',
                              'Avg_Order', 'Units_Sold', 'MoM_Change', 'MoM_Pct']].copy()
monthly_summary['Partial_Month'] = (monthly_summary['Month'] == 4).astype(int)

# ── cohort_retention & sales_forecast (unchanged) ─────────────────────────────
# sales_forecast Month is already Python date from Cell 30 — MySQL DATE ✓
cohort_retention  = retention_flat.copy()
sales_forecast    = forecast_df.copy()

# ── Summary log ────────────────────────────────────────────────────────────────
for name, frame in [('dim_date', dim_date),('dim_customer', dim_customer),
                    ('dim_product', dim_product),('dim_payment', dim_payment),
                    ('fact_sales', fact_sales),('customer_segments', customer_segments),
                    ('monthly_summary', monthly_summary),
                    ('cohort_retention', cohort_retention),('sales_forecast', sales_forecast)]:
    log.info(f'  {name:<28}  {len(frame):>6,} rows ready')


[INFO] dim_date built  ->  105 rows  (expect 105, date range 2024-01-01 to 2024-04-14)
[INFO] dim_product built  ->  25 rows  (expect 25, Product_Key unique: 25)
[INFO] fact_sales built  ->  2,705 rows
[INFO]   dim_date                         105 rows ready
[INFO]   dim_customer                   1,000 rows ready
[INFO]   dim_product                       25 rows ready
[INFO]   dim_payment                        4 rows ready
[INFO]   fact_sales                     2,705 rows ready
[INFO]   customer_segments              1,000 rows ready
[INFO]   monthly_summary                    4 rows ready
[INFO]   cohort_retention                  16 rows ready
[INFO]   sales_forecast                     3 rows ready


In [46]:
# ── LOAD ──────────────────────────────────────────────────────────────────────
def load(engine, tables: dict):
    for name, frame in tables.items():
        frame.to_sql(name, con=engine, if_exists='replace',
                     index=False, chunksize=500)
        log.info(f'Loaded  {name:<35}  {len(frame):>6,} rows')
    log.info('ETL complete — all tables loaded to MySQL.')

load(engine, {
    'dim_date'               : dim_date,
    'dim_customer'           : dim_customer,
    'dim_product'            : dim_product,
    'dim_payment'            : dim_payment,
    'fact_sales'             : fact_sales,
    'customer_segments'      : customer_segments,
    'monthly_revenue_summary': monthly_summary,
    'cohort_retention'       : cohort_retention,
    'sales_forecast'         : sales_forecast,
})

[INFO] Loaded  dim_date                                105 rows
[INFO] Loaded  dim_customer                          1,000 rows
[INFO] Loaded  dim_product                              25 rows
[INFO] Loaded  dim_payment                               4 rows
[INFO] Loaded  fact_sales                            2,705 rows
[INFO] Loaded  customer_segments                     1,000 rows
[INFO] Loaded  monthly_revenue_summary                   4 rows
[INFO] Loaded  cohort_retention                         16 rows
[INFO] Loaded  sales_forecast                            3 rows
[INFO] ETL complete — all tables loaded to MySQL.


---
## 13. SQL Analytical Queries

In [ ]:
def run_query(title: str, sql: str) -> pd.DataFrame:
    result = pd.read_sql(sql, engine)
    print(f'\n-- {title} --')
    print(result.to_string(index=False))
    return result

run_query('Revenue by Category', """
    SELECT p.Product_Category,
    ROUND(SUM(f.Net_Amount), 2)   AS Revenue,
    COUNT(f.Transaction_ID)        AS Transactions,
    ROUND(AVG(f.Net_Amount), 2)   AS Avg_Order,
    ROUND(SUM(f.Net_Amount) / SUM(SUM(f.Net_Amount)) OVER () * 100, 2) AS Revenue_Share_Pct
    FROM fact_sales f JOIN dim_product p USING (Product_Name)
    GROUP BY p.Product_Category ORDER BY Revenue DESC;""")

run_query('Revenue & CLV by Customer Segment', """
    SELECT cs.Segment,
           COUNT(DISTINCT f.Customer_ID)  AS Customers,
           ROUND(SUM(f.Net_Amount), 2)    AS Total_Revenue,
           ROUND(AVG(f.Net_Amount), 2)    AS Avg_Transaction,
           ROUND(AVG(cs.CLV), 2)          AS Avg_CLV,
           ROUND(AVG(cs.Recency), 1)      AS Avg_Recency_Days,
           ROUND(AVG(cs.Frequency), 2)    AS Avg_Frequency
    FROM fact_sales f JOIN customer_segments cs USING (Customer_ID)
    GROUP BY cs.Segment ORDER BY Total_Revenue DESC;""")

run_query('Discount & Promotion Impact', """
    SELECT CASE Promotion_Flag WHEN 1 THEN 'Promotion' ELSE 'No Promotion' END AS Status,
           COUNT(*)                            AS Transactions,
           ROUND(AVG(Net_Amount), 2)           AS Avg_Net_Revenue,
           ROUND(AVG(Gross_Amount), 2)         AS Avg_Gross_Revenue,
           ROUND(AVG(Discount_Rate)*100, 2)    AS Avg_Discount_Pct,
           ROUND(SUM(Discount_Amount), 2)      AS Total_Discount_Given,
           ROUND(AVG(Net_Profit), 2)           AS Avg_Net_Profit
    FROM fact_sales GROUP BY Promotion_Flag;""")

run_query('Peak Shopping Hours (Top 5)', """
    SELECT Hour, COUNT(*) AS Transactions,
           ROUND(SUM(Net_Amount), 2) AS Revenue,
           ROUND(AVG(Net_Amount), 2) AS Avg_Order
    FROM fact_sales
    GROUP BY Hour ORDER BY Transactions DESC LIMIT 5;""")

run_query('Monthly Revenue with MoM Change', """
    SELECT Year, Month, Period,
           ROUND(Revenue, 2) AS Revenue, Transactions,
           ROUND(MoM_Change, 2) AS MoM_Change, MoM_Pct
    FROM monthly_revenue_summary ORDER BY Year, Month;""")

run_query('CLV Summary by Segment', """
    SELECT Segment, COUNT(*) AS Customers,
           ROUND(MIN(CLV), 2) AS Min_CLV,
           ROUND(AVG(CLV), 2) AS Avg_CLV,
           ROUND(MAX(CLV), 2) AS Max_CLV,
           ROUND(SUM(CLV), 2) AS Total_CLV
    FROM customer_segments
    GROUP BY Segment ORDER BY Avg_CLV DESC;""")

run_query('3-Month Sales Forecast (SARIMA)', """
    SELECT Month, Forecast_Revenue, CI_Lower, CI_Upper
    FROM sales_forecast;""")


-- Revenue by Category --
Product_Category    Revenue  Transactions  Avg_Order  Revenue_Share_Pct
      Home Decor 1670762.22          2705     617.66               20.0
     Electronics 1670762.22          2705     617.66               20.0
        Clothing 1670762.22          2705     617.66               20.0
           Books 1670762.22          2705     617.66               20.0
          Beauty 1670762.22          2705     617.66               20.0

-- Revenue & CLV by Customer Segment --
         Segment  Customers  Total_Revenue  Avg_Transaction  Avg_CLV  Avg_Recency_Days  Avg_Frequency
      High-Value        277      845692.01           697.19  3146.72              18.7           4.71
  Frequent Buyer        449      571598.74           549.09  1370.71              19.7           2.62
Occasional Buyer        274      253471.47           562.02  1106.19              66.7           1.97

-- Discount & Promotion Impact --
      Status  Transactions  Avg_Net_Revenue  Avg_Gross_Re

,Month,Forecast_Revenue,CI_Lower,CI_Upper
0,2024-04-01,501444.78,420747.43,582142.13
1,2024-05-01,501444.78,387321.50,615568.06
2,2024-06-01,501444.78,361672.87,641216.69


---
## 14. Executive Summary & Recommendations

In [48]:
top_category  = cat['Revenue'].idxmax()
low_category  = cat['Revenue'].idxmin()
top_segment   = cluster_profile['Total_Revenue'].idxmax()
avg_clv       = rfm['CLV'].mean()
next_forecast = fcast_mean.iloc[0]

print('=' * 62)
print('           TRENDMART — EXECUTIVE SUMMARY')
print('=' * 62)
print(f'  Total Net Revenue         : ${total_revenue:>12,.2f}')
print(f'  Total Gross Revenue       : ${df["Gross_Amount"].sum():>12,.2f}')
print(f'  Total Discounts Given     : ${total_discount:>12,.2f}')
print(f'  Avg Order Value (AOV)     : ${aov:>12,.2f}')
print(f'  Revenue per Customer      : ${rev_per_customer:>12,.2f}')
print(f'  Repeat Purchase Rate      : {repeat_rate:>12.2f}%')
print(f'  Top Revenue Category      :  {top_category}')
print(f'  Lowest Revenue Category   :  {low_category}')
print(f'  Peak Shopping Hour        :  {peak_hour}:00')
print(f'  Avg Customer CLV          : ${avg_clv:>12,.2f}')
print(f'  Next-Month Revenue Fcst   : ${next_forecast:>12,.2f}')
print('=' * 62)
print()
print('STRATEGIC RECOMMENDATIONS')
print('-' * 62)
recs = [
    ('INVENTORY'  , f'Prioritise {top_category}; reassess {low_category} assortment.'),
    ('PROMOTIONS' , 'Cap discounts at 10% to protect margins; focus on bulk orders.'),
    ('TIMING'     , f'Schedule flash sales at {peak_hour}:00 for peak traffic.'),
    ('CRM'        , f'Activate VIP rewards for {top_segment} (highest CLV segment).'),
    ('RETENTION'  , 'Re-engage Occasional Buyers via personalised email campaigns.'),
    ('FORECASTING', f'Pre-position ${next_forecast:,.0f} of inventory for next month.'),
]
for area, rec in recs:
    print(f'  [{area:<12}] {rec}')

           TRENDMART — EXECUTIVE SUMMARY
  Total Net Revenue         : $1,670,762.22
  Total Gross Revenue       : $1,782,148.00
  Total Discounts Given     : $  111,385.78
  Avg Order Value (AOV)     : $      617.66
  Revenue per Customer      : $    1,670.76
  Repeat Purchase Rate      :        78.30%
  Top Revenue Category      :  Clothing
  Lowest Revenue Category   :  Beauty
  Peak Shopping Hour        :  12:00
  Avg Customer CLV          : $    1,670.76
  Next-Month Revenue Fcst   : $  501,444.78

STRATEGIC RECOMMENDATIONS
--------------------------------------------------------------
  [INVENTORY   ] Prioritise Clothing; reassess Beauty assortment.
  [PROMOTIONS  ] Cap discounts at 10% to protect margins; focus on bulk orders.
  [TIMING      ] Schedule flash sales at 12:00 for peak traffic.
  [CRM         ] Activate VIP rewards for High-Value (highest CLV segment).
  [RETENTION   ] Re-engage Occasional Buyers via personalised email campaigns.
  [FORECASTING ] Pre-position $501,4